In [7]:
%pip install pyspark==4.0.1 findspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col, when

spark = (
    SparkSession.builder
    .appName("big-data-programming-2.3.1")
    .master("local[*]")
    .getOrCreate()
)

In [13]:
df = spark.read.csv(
    "./input/weatherData.csv",
    header=True,
    inferSchema=True,
)

df.show()

+-----------+---------+-----------------------+-----------------------+-----------------------+------------------------+-----------------------------+-----------------------------+------------------------------+---------------------+---------------------+----------------------+-------------+-----------------------+-------------------------+-------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------+-------------------+
|location_id|     date|weather_code (wmo code)|temperature_2m_max (°C)|temperature_2m_min (°C)|temperature_2m_mean (°C)|apparent_temperature_max (°C)|apparent_temperature_min (°C)|apparent_temperature_mean (°C)|daylight_duration (s)|sunshine_duration (s)|precipitation_sum (mm)|rain_sum (mm)|precipitation_hours (h)|wind_speed_10m_max (km/h)|wind_gusts_10m_max (km/h)|wind_direction_10m_dominant (°)|shortwave_radiation_sum (MJ/m²)|et0_fao_evapotranspiration (mm)|            sunrise|           

In [14]:
df = df.withColumn("month", split(col("date"), "/").getItem(0))
df = df.select("month", "shortwave_radiation_sum (MJ/m²)")
df.show()

+-----+-------------------------------+
|month|shortwave_radiation_sum (MJ/m²)|
+-----+-------------------------------+
|    1|                          20.92|
|    1|                          17.71|
|    1|                          17.76|
|    1|                           16.5|
|    1|                          23.61|
|    1|                          21.48|
|    1|                          19.38|
|    1|                           13.0|
|    1|                          16.89|
|    1|                          20.52|
|    1|                          19.88|
|    1|                          19.56|
|    1|                          18.41|
|    1|                          19.46|
|    1|                          21.05|
|    1|                          19.46|
|    1|                          16.88|
|    1|                          15.79|
|    1|                           19.4|
|    1|                          22.72|
+-----+-------------------------------+
only showing top 20 rows


In [15]:
df = df.withColumn("shortwave_radiation_over_15", when(col("shortwave_radiation_sum (MJ/m²)") > 15, 1).otherwise(0))
df.show()

+-----+-------------------------------+---------------------------+
|month|shortwave_radiation_sum (MJ/m²)|shortwave_radiation_over_15|
+-----+-------------------------------+---------------------------+
|    1|                          20.92|                          1|
|    1|                          17.71|                          1|
|    1|                          17.76|                          1|
|    1|                           16.5|                          1|
|    1|                          23.61|                          1|
|    1|                          21.48|                          1|
|    1|                          19.38|                          1|
|    1|                           13.0|                          0|
|    1|                          16.89|                          1|
|    1|                          20.52|                          1|
|    1|                          19.88|                          1|
|    1|                          19.56|         

In [17]:
percentage_df = df.groupBy("month").avg("shortwave_radiation_over_15")
percentage_df.show()

+-----+--------------------------------+
|month|avg(shortwave_radiation_over_15)|
+-----+--------------------------------+
|    7|              0.8934971838197645|
|   11|              0.6132275132275132|
|    3|              0.9624850657108721|
|    8|              0.8848779655231268|
|    5|              0.8500199123855038|
|    6|              0.8841294565593631|
|    9|              0.8591710758377425|
|    1|              0.7885304659498208|
|   10|              0.7610513739545998|
|    4|              0.9556378600823046|
|   12|               0.592763270182625|
|    2|              0.8976240391334731|
+-----+--------------------------------+



In [19]:
spark.stop()